In [1]:
import sys
print(sys.executable)

C:\Users\ROG\.conda\envs\detr-gpu\python.exe


In [3]:
import sys
!{sys.executable} -m pip install -U ultralytics opencv-python pyyaml pandas matplotlib

In [5]:
data_yaml = """
path: C:/Users/ROG/Desktop/Datasets_with_annotation_object_detection/Merged_dataset
train: images/train
val: images/val
test: images/test

names:
  0: Interphase
  1: Prophase
  2: Lateprophase
  3: Metaphase
  4: Anaphase
  5: Lateanaphase
  6: Telophase
"""

yaml_path = r"C:\Users\ROG\Desktop\Datasets_with_annotation_object_detection\Merged_dataset\data.yaml"

with open(yaml_path, "w", encoding="utf-8") as f:
    f.write(data_yaml)

print("Saved:", yaml_path)

Saved: C:\Users\ROG\Desktop\Datasets_with_annotation_object_detection\Merged_dataset\data.yaml


In [7]:
import os, glob

base = r"C:\Users\ROG\Desktop\Datasets_with_annotation_object_detection\Merged_dataset"

for p in [
    os.path.join(base, "images", "train"),
    os.path.join(base, "images", "val"),
    os.path.join(base, "images", "test"),
    os.path.join(base, "labels", "train"),
    os.path.join(base, "labels", "val"),
    os.path.join(base, "labels", "test"),
]:
    print(p, "->", os.path.exists(p))

for split in ["train", "val", "test"]:
    imgs = []
    for ext in ("*.jpg", "*.jpeg", "*.png", "*.bmp", "*.tif", "*.tiff"):
        imgs.extend(glob.glob(os.path.join(base, "images", split, ext)))
    lbls = glob.glob(os.path.join(base, "labels", split, "*.txt"))
    print(split, "images:", len(imgs), "labels:", len(lbls))

C:\Users\ROG\Desktop\Datasets_with_annotation_object_detection\Merged_dataset\images\train -> True
C:\Users\ROG\Desktop\Datasets_with_annotation_object_detection\Merged_dataset\images\val -> True
C:\Users\ROG\Desktop\Datasets_with_annotation_object_detection\Merged_dataset\images\test -> True
C:\Users\ROG\Desktop\Datasets_with_annotation_object_detection\Merged_dataset\labels\train -> True
C:\Users\ROG\Desktop\Datasets_with_annotation_object_detection\Merged_dataset\labels\val -> True
C:\Users\ROG\Desktop\Datasets_with_annotation_object_detection\Merged_dataset\labels\test -> True
train images: 364 labels: 364
val images: 91 labels: 91
test images: 47 labels: 47


In [9]:
from ultralytics import YOLO

model = YOLO("yolov8n-obb.pt")   # or another available -obb checkpoint in your installed version

In [11]:
from ultralytics import YOLO
from pathlib import Path
import os
import traceback

RUN_PROJECT = "runs_obb"
RUN_NAME = "obb_baseline_832_ep100"

model = YOLO("yolov8n-obb.pt")

try:
    results = model.train(
        data=yaml_path,
        epochs=100,
        imgsz=832,
        batch=8,
        device=0,
        project=RUN_PROJECT,
        name=RUN_NAME,
        save=True,
        exist_ok=True,
        verbose=True
    )
except Exception:
    print("Training crashed:")
    traceback.print_exc()

weights_dir = Path(RUN_PROJECT) / RUN_NAME / "weights"
best_path = weights_dir / "best.pt"
last_path = weights_dir / "last.pt"

print("\nWeights directory:", weights_dir)
print("Weights dir exists:", weights_dir.exists())

if weights_dir.exists():
    print("Available files:", os.listdir(weights_dir))

model_path = None
if best_path.exists():
    model_path = best_path
elif last_path.exists():
    model_path = last_path

if model_path is not None:
    print(f"\nUsing weights: {model_path}")
    eval_model = YOLO(str(model_path))
    metrics = eval_model.val(data=yaml_path)

    p = metrics.box.p
    r = metrics.box.r
    map50 = metrics.box.map50
    map5095 = metrics.box.map
    f1 = 2 * p * r / (p + r + 1e-16)

    print(f"Precision   : {p}")
    print(f"Recall      : {r}")
    print(f"mAP50       : {map50}")
    print(f"mAP50-95    : {map5095}")
    print(f"F1 Score    : {f1}")
else:
    print("\nNo best.pt or last.pt found. Training likely failed before checkpoint saving completed.")

Ultralytics 8.4.21  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\ROG\Desktop\Datasets_with_annotation_object_detection\Merged_dataset\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=832, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-obb.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=obb_baseline_832_ep100, nbs=64, n

In [13]:
from ultralytics import YOLO
from pathlib import Path
import traceback
import os

RUN_PROJECT = "runs_obb"
RUN_NAME = "obb_debug_832_ep100"

model = YOLO("yolov8n-obb.pt")   # use an OBB pretrained model

try:
    results = model.train(
        data=yaml_path,
        epochs=100,
        imgsz=832,
        batch=8,
        device=0,
        project=RUN_PROJECT,
        name=RUN_NAME,
        save=True,
        exist_ok=True,
        verbose=True
    )
    print("Training finished successfully.")

except Exception as e:
    print("\nTRAINING FAILED:")
    traceback.print_exc()

run_dir = Path(RUN_PROJECT) / RUN_NAME
weights_dir = run_dir / "weights"

print("\nRun dir exists:", run_dir.exists())
print("Weights dir exists:", weights_dir.exists())
if weights_dir.exists():
    print("Weights files:", os.listdir(weights_dir))

Ultralytics 8.4.21  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\ROG\Desktop\Datasets_with_annotation_object_detection\Merged_dataset\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=832, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-obb.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=obb_debug_832_ep100, nbs=64, nms=

In [15]:
best_pt = r"C:\Users\ROG\runs\obb\runs_obb\obb_debug_832_ep100\weights\best.pt"

In [17]:
from ultralytics import YOLO
import os

best_pt = r"C:\Users\ROG\runs\obb\runs_obb\obb_debug_832_ep100\weights\best.pt"

print("best.pt exists:", os.path.isfile(best_pt))

best_model = YOLO(best_pt)
print("Model loaded successfully")

best.pt exists: True
Model loaded successfully


In [19]:
from pathlib import Path
import os

run_dir = Path(r"C:\Users\ROG\runs\obb\runs_obb\obb_debug_832_ep100")
weights_dir = run_dir / "weights"

print("Run dir exists:", run_dir.exists(), run_dir)
print("Weights dir exists:", weights_dir.exists(), weights_dir)

if weights_dir.exists():
    print("Weights files:", os.listdir(weights_dir))

Run dir exists: True C:\Users\ROG\runs\obb\runs_obb\obb_debug_832_ep100
Weights dir exists: True C:\Users\ROG\runs\obb\runs_obb\obb_debug_832_ep100\weights
Weights files: ['best.pt', 'last.pt']


In [21]:
from ultralytics import YOLO

best_pt = r"C:\Users\ROG\runs\obb\runs_obb\obb_debug_832_ep100\weights\best.pt"
best_model = YOLO(best_pt)

conf_list = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50]

for conf in conf_list:
    best_model.predict(
        source=r"C:\Users\ROG\Desktop\Datasets_with_annotation_object_detection\Merged_dataset\images\test",
        conf=conf,
        save=True,
        save_txt=True,
        project=r"C:\Users\ROG\runs_obb_thresholds",
        name=f"pred_conf_{str(conf).replace('.', '_')}",
        exist_ok=True
    )
    print("Done:", conf)


image 1/47 C:\Users\ROG\Desktop\Datasets_with_annotation_object_detection\Merged_dataset\images\test\A_027934b8-task-936_merged.png: 480x832 27 Interphases, 13 Prophases, 4 Lateprophases, 59.4ms
image 2/47 C:\Users\ROG\Desktop\Datasets_with_annotation_object_detection\Merged_dataset\images\test\A_12a62314-task-918_merged.png: 480x832 32 Interphases, 15 Prophases, 1 Lateprophase, 35.6ms
image 3/47 C:\Users\ROG\Desktop\Datasets_with_annotation_object_detection\Merged_dataset\images\test\A_131eae8f-task-885_merged.png: 480x832 107 Interphases, 16 Prophases, 4 Metaphases, 3 Anaphases, 35.4ms
image 4/47 C:\Users\ROG\Desktop\Datasets_with_annotation_object_detection\Merged_dataset\images\test\A_14d4b2bd-task-958_merged.png: 480x832 27 Interphases, 1 Prophase, 35.6ms
image 5/47 C:\Users\ROG\Desktop\Datasets_with_annotation_object_detection\Merged_dataset\images\test\A_150ac7b3-task-943_merged.png: 480x832 26 Interphases, 2 Prophases, 36.5ms
image 6/47 C:\Users\ROG\Desktop\Datasets_with_annot

In [25]:
from ultralytics import YOLO

model = YOLO("yolov8n-obb.pt")

results = model.train(
    data=yaml_path,
    epochs=100,
    imgsz=832,
    batch=8,
    device=0,
    project=r"C:\Users\ROG\runs_obb",
    name="obb_debug_832_ep100",
    save=True,
    exist_ok=True,
    verbose=True
)

print("Actual save dir:", results.save_dir)

Ultralytics 8.4.21  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\ROG\Desktop\Datasets_with_annotation_object_detection\Merged_dataset\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=832, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-obb.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=obb_debug_832_ep100, nbs=64, nms=

In [27]:
from ultralytics import YOLO
import os

best_pt = r"C:\Users\ROG\runs\obb\runs_obb\obb_debug_832_ep100\weights\best.pt"
print(os.path.isfile(best_pt))

best_model = YOLO(best_pt)
print("Model loaded successfully")

True
Model loaded successfully


In [29]:
metrics = best_model.val(data=yaml_path, split="val")
print(metrics)

Ultralytics 8.4.21  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 6144MiB)
YOLOv8n-obb summary (fused): 82 layers, 3,078,584 parameters, 0 gradients, 8.3 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 2059.8422.8 MB/s, size: 3507.9 KB)
val: Scanning C:\Users\ROG\Desktop\Datasets_with_annotation_object_detection\Merged_dataset\labels\val.cache... 91 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 91/91  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 1.2s/it 7.0s1.0ss
                   all         91       2188      0.801      0.547      0.662      0.553
            Interphase         87       1106       0.74      0.638      0.758      0.679
              Prophase         84        709      0.611      0.553      0.613      0.543
          Lateprophase         47        221      0.716      0.616      0.738      0.621
             Metaphase         42         94      0.933     

In [31]:
import os
from ultralytics import YOLO

best_paths = [
    r"C:\Users\ROG\runs\obb\runs_obb\obb_debug_832_ep100\weights\best.pt",
]

for best_pt in best_paths:
    if os.path.isfile(best_pt):
        model = YOLO(best_pt)
        metrics = model.val(data=yaml_path, split="val")
        print(best_pt)
        print(metrics)

Ultralytics 8.4.21  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 6144MiB)
YOLOv8n-obb summary (fused): 82 layers, 3,078,584 parameters, 0 gradients, 8.3 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 2738.3118.7 MB/s, size: 3234.8 KB)
val: Scanning C:\Users\ROG\Desktop\Datasets_with_annotation_object_detection\Merged_dataset\labels\val.cache... 91 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 91/91  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 1.2s/it 6.9s1.1ss
                   all         91       2188      0.801      0.547      0.662      0.553
            Interphase         87       1106       0.74      0.638      0.758      0.679
              Prophase         84        709      0.611      0.553      0.613      0.543
          Lateprophase         47        221      0.716      0.616      0.738      0.621
             Metaphase         42         94      0.933     

In [33]:
import os
import json
import pandas as pd
from ultralytics import YOLO

# Dataset YAML
yaml_path = r"C:\Users\ROG\Desktop\Datasets_with_annotation_object_detection\Merged_dataset\data.yaml"

# Best model from your successful run
best_pt = r"C:\Users\ROG\runs\obb\runs_obb\obb_debug_832_ep100\weights\best.pt"

# Test images
test_source = r"C:\Users\ROG\Desktop\Datasets_with_annotation_object_detection\Merged_dataset\images\test"

print("yaml exists:", os.path.isfile(yaml_path))
print("best.pt exists:", os.path.isfile(best_pt))
print("test folder exists:", os.path.isdir(test_source))

yaml exists: True
best.pt exists: True
test folder exists: True


In [35]:
best_model = YOLO(best_pt)
print("Model loaded successfully")

Model loaded successfully


In [37]:
metrics = best_model.val(data=yaml_path, split="val")
print(metrics)

Ultralytics 8.4.21  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 6144MiB)
YOLOv8n-obb summary (fused): 82 layers, 3,078,584 parameters, 0 gradients, 8.3 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 2273.7171.1 MB/s, size: 3577.7 KB)
val: Scanning C:\Users\ROG\Desktop\Datasets_with_annotation_object_detection\Merged_dataset\labels\val.cache... 91 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 91/91  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 1.3s/it 8.1s1.0ss
                   all         91       2188      0.801      0.547      0.662      0.553
            Interphase         87       1106       0.74      0.638      0.758      0.679
              Prophase         84        709      0.611      0.553      0.613      0.543
          Lateprophase         47        221      0.716      0.616      0.738      0.621
             Metaphase         42         94      0.933     

In [39]:
conf_list = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50]
rows = []

for conf in conf_list:
    m = best_model.val(
        data=yaml_path,
        split="val",
        conf=conf,
        imgsz=832,
        plots=False,
        save_json=False,
        verbose=False
    )

    row = {
        "conf": conf,
        "precision_B": getattr(m.box, "mp", None) if hasattr(m, "box") else None,
        "recall_B": getattr(m.box, "mr", None) if hasattr(m, "box") else None,
        "map50_B": getattr(m.box, "map50", None) if hasattr(m, "box") else None,
        "map50_95_B": getattr(m.box, "map", None) if hasattr(m, "box") else None,
    }

    if row["precision_B"] is not None and row["recall_B"] is not None:
        p = row["precision_B"]
        r = row["recall_B"]
        row["f1_B"] = 2 * p * r / (p + r + 1e-9)
    else:
        row["f1_B"] = None

    rows.append(row)
    print(row)

threshold_df = pd.DataFrame(rows)
threshold_df

Ultralytics 8.4.21  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 6144MiB)
val: Fast image access  (ping: 0.10.0 ms, read: 1211.4512.0 MB/s, size: 3785.5 KB)
val: Scanning C:\Users\ROG\Desktop\Datasets_with_annotation_object_detection\Merged_dataset\labels\val.cache... 91 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 91/91  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 2.3it/s 2.7s0.3ss
                   all         91       2188      0.801      0.547       0.65      0.545
Speed: 1.7ms preprocess, 7.1ms inference, 0.0ms loss, 4.9ms postprocess per image
{'conf': 0.05, 'precision_B': np.float64(0.80093557338269), 'recall_B': np.float64(0.5469009321936743), 'map50_B': np.float64(0.650316223444653), 'map50_95_B': np.float64(0.5450401068314052), 'f1_B': np.float64(0.6499785538866476)}
Ultralytics 8.4.21  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 61

,conf,precision_B,recall_B,map50_B,map50_95_B,f1_B
0,0.05,0.800936,0.546901,0.650316,0.545040,0.649979
1,0.10,0.800936,0.546901,0.644021,0.541574,0.649979
2,0.15,0.800936,0.546901,0.642120,0.541667,0.649979
3,0.20,0.800936,0.546901,0.637918,0.539199,0.649979
4,0.25,0.658078,0.546901,0.634602,0.537625,0.597361
5,0.30,0.658078,0.546901,0.630591,0.535020,0.597361
6,0.35,0.658078,0.546901,0.627582,0.534407,0.597361
7,0.40,0.658078,0.546901,0.624406,0.531638,0.597361
8,0.50,0.666243,0.546878,0.621644,0.530785,0.600688


In [41]:
threshold_df = threshold_df.sort_values(by="f1_B", ascending=False)
best_conf = float(threshold_df.iloc[0]["conf"])
print("Best threshold by F1:", best_conf)
threshold_df.head(10)

Best threshold by F1: 0.05


,conf,precision_B,recall_B,map50_B,map50_95_B,f1_B
0,0.05,0.800936,0.546901,0.650316,0.545040,0.649979
1,0.10,0.800936,0.546901,0.644021,0.541574,0.649979
2,0.15,0.800936,0.546901,0.642120,0.541667,0.649979
3,0.20,0.800936,0.546901,0.637918,0.539199,0.649979
8,0.50,0.666243,0.546878,0.621644,0.530785,0.600688
4,0.25,0.658078,0.546901,0.634602,0.537625,0.597361
5,0.30,0.658078,0.546901,0.630591,0.535020,0.597361
6,0.35,0.658078,0.546901,0.627582,0.534407,0.597361
7,0.40,0.658078,0.546901,0.624406,0.531638,0.597361


In [43]:
pred_results = best_model.predict(
    source=test_source,
    conf=best_conf,
    imgsz=832,
    save=True,
    save_txt=True,
    project=r"C:\Users\ROG\runs_obb_thresholds",
    name=f"best_conf_{str(best_conf).replace('.', '_')}",
    exist_ok=True
)

print("Saved thresholded predictions.")


image 1/47 C:\Users\ROG\Desktop\Datasets_with_annotation_object_detection\Merged_dataset\images\test\A_027934b8-task-936_merged.png: 480x832 29 Interphases, 14 Prophases, 4 Lateprophases, 44.0ms
image 2/47 C:\Users\ROG\Desktop\Datasets_with_annotation_object_detection\Merged_dataset\images\test\A_12a62314-task-918_merged.png: 480x832 35 Interphases, 18 Prophases, 1 Lateprophase, 36.3ms
image 3/47 C:\Users\ROG\Desktop\Datasets_with_annotation_object_detection\Merged_dataset\images\test\A_131eae8f-task-885_merged.png: 480x832 135 Interphases, 19 Prophases, 4 Metaphases, 3 Anaphases, 33.2ms
image 4/47 C:\Users\ROG\Desktop\Datasets_with_annotation_object_detection\Merged_dataset\images\test\A_14d4b2bd-task-958_merged.png: 480x832 30 Interphases, 1 Prophase, 31.3ms
image 5/47 C:\Users\ROG\Desktop\Datasets_with_annotation_object_detection\Merged_dataset\images\test\A_150ac7b3-task-943_merged.png: 480x832 28 Interphases, 2 Prophases, 31.2ms
image 6/47 C:\Users\ROG\Desktop\Datasets_with_annot

In [45]:
model_s = YOLO("yolov8s-obb.pt")

results_s = model_s.train(
    data=yaml_path,
    epochs=150,
    imgsz=832,
    batch=8,
    device=0,
    project=r"C:\Users\ROG\runs_obb",
    name="obb_s_832_ep150",
    save=True,
    exist_ok=True,
    verbose=True
)

print("Actual save dir:", results_s.save_dir)

Ultralytics 8.4.21  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\ROG\Desktop\Datasets_with_annotation_object_detection\Merged_dataset\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=832, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s-obb.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=obb_s_832_ep150, nbs=64, nms=Fals

In [47]:
best_pt_s = os.path.join(str(results_s.save_dir), "weights", "best.pt")
print("best_pt_s exists:", os.path.isfile(best_pt_s))

best_model_s = YOLO(best_pt_s)
metrics_s = best_model_s.val(data=yaml_path, split="val")
print(metrics_s)

best_pt_s exists: True
Ultralytics 8.4.21  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 6144MiB)
YOLOv8s-obb summary (fused): 82 layers, 11,414,280 parameters, 0 gradients, 29.4 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 2301.6580.7 MB/s, size: 3507.9 KB)
val: Scanning C:\Users\ROG\Desktop\Datasets_with_annotation_object_detection\Merged_dataset\labels\val.cache... 91 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 91/91  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 1.1s/it 6.4s0.8ss
                   all         91       2188      0.758       0.79      0.848      0.731
            Interphase         87       1106      0.772      0.741      0.815       0.74
              Prophase         84        709      0.628      0.742       0.73      0.657
          Lateprophase         47        221      0.681      0.783      0.792      0.695
             Metaphase         42  

In [49]:
conf_list = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50]
rows_s = []

for conf in conf_list:
    m = best_model_s.val(
        data=yaml_path,
        split="val",
        conf=conf,
        imgsz=832,
        plots=False,
        save_json=False,
        verbose=False
    )

    row = {
        "conf": conf,
        "precision_B": getattr(m.box, "mp", None) if hasattr(m, "box") else None,
        "recall_B": getattr(m.box, "mr", None) if hasattr(m, "box") else None,
        "map50_B": getattr(m.box, "map50", None) if hasattr(m, "box") else None,
        "map50_95_B": getattr(m.box, "map", None) if hasattr(m, "box") else None,
    }

    if row["precision_B"] is not None and row["recall_B"] is not None:
        p = row["precision_B"]
        r = row["recall_B"]
        row["f1_B"] = 2 * p * r / (p + r + 1e-9)
    else:
        row["f1_B"] = None

    rows_s.append(row)

threshold_df_s = pd.DataFrame(rows_s).sort_values(by="f1_B", ascending=False)
threshold_df_s.head(10)

Ultralytics 8.4.21  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 6144MiB)
val: Fast image access  (ping: 0.10.0 ms, read: 1210.6686.7 MB/s, size: 3234.8 KB)
val: Scanning C:\Users\ROG\Desktop\Datasets_with_annotation_object_detection\Merged_dataset\labels\val.cache... 91 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 91/91  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 2.1it/s 2.9s0.4ss
                   all         91       2188      0.758       0.79      0.847      0.733
Speed: 2.6ms preprocess, 9.1ms inference, 0.0ms loss, 4.6ms postprocess per image
Ultralytics 8.4.21  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 6144MiB)
val: Fast image access  (ping: 0.10.0 ms, read: 1765.3138.7 MB/s, size: 3577.7 KB)
val: Scanning C:\Users\ROG\Desktop\Datasets_with_annotation_object_detection\Merged_dataset\labels\val.cache... 91 images, 0 backgrounds, 0 co

,conf,precision_B,recall_B,map50_B,map50_95_B,f1_B
0,0.05,0.758410,0.789630,0.846767,0.733209,0.773705
1,0.10,0.758410,0.789630,0.838266,0.727948,0.773705
2,0.15,0.758410,0.789630,0.835170,0.727642,0.773705
3,0.20,0.758410,0.789630,0.836399,0.729409,0.773705
4,0.25,0.758410,0.789630,0.836230,0.730139,0.773705
5,0.30,0.758410,0.789630,0.833136,0.728634,0.773705
6,0.35,0.758410,0.789630,0.828374,0.726084,0.773705
8,0.50,0.789678,0.738635,0.799365,0.697924,0.763304
7,0.40,0.790031,0.733362,0.803983,0.700687,0.760643
